# 02 Plot

Loads `../output/full_dataset.csv` and generates publication figures.

**GO slim categories** (`Function_Slim`, `Location_Slim`) are the primary grouping for all
category plots (Figs 3–9).  
**Raw GO term filtering** lives in its own section at the bottom — kept separate from the slim categories.

**Output:** `../output/figures/`

| Figure | Description |
|---|---|
| `fig1_pllps_distribution` | pLLPS histogram (count) + density/KDE by class |
| `fig2_membrane_vs_nonmembrane` | Histogram overlay + violin/box: membrane vs non-membrane |
| `fig3_pllps_by_go_slim_function` | Faceted pLLPS histograms by top GO slim function (vs all-protein background) |
| `fig4_by_location` | pLLPS violin/box by GO slim location — all proteins, top 10 |
| `fig5_2d_kde` | 2D KDE: pLLPS vs protein length and DPR count; violin/box vs TMD count |
| `fig6_membrane_by_function` | Membrane proteins: pLLPS violin/box by GO slim function |
| `fig7_membrane_by_location` | Membrane proteins: pLLPS violin/box by GO slim location |
| `fig8_pct_above_cutoff_function` | % proteins above p(LLPS) cutoff by GO slim function (membrane) |
| `fig9_pct_above_cutoff_location` | % proteins above p(LLPS) cutoff by GO slim location (membrane) |

In [ ]:
from ast import literal_eval
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_style("whitegrid")
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "xtick.labelsize": 9, "ytick.labelsize": 9})

DATA    = Path("../output/full_dataset.csv")
FIG_DIR = Path("../output/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

CUTOFF = 0.6   # pLLPS threshold for enrichment plots
TOP_N  = 10    # categories shown in violin/box plots

# Okabe-Ito colorblind-friendly palette
C = {
    "Membrane":     "#0072B2",
    "Non-Membrane": "#E69F00",
    "Cytosolic":    "#009E73",
    "High":         "#e74c3c",
    "Medium":       "#f39c12",
    "Low":          "#3498db",
}

In [ ]:
df = pd.read_csv(DATA)

df["TMD_count"]   = pd.to_numeric(df["TMD_count"],  errors="coerce").fillna(0).astype(int)
df["Is_Membrane"] = df["TMD_count"] > 0
df["Length"]      = pd.to_numeric(df["Length"],      errors="coerce")

def _as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("["):
            try:
                parsed = literal_eval(text)
                return parsed if isinstance(parsed, list) else []
            except (ValueError, SyntaxError):
                return []
    return []

# -- GO slim columns: primary grouping for all category plots -----------------
df["Function_Slim"]      = df["Function_Slim"].apply(_as_list)
df["Location_Slim"]      = df["Location_Slim"].apply(_as_list)
df["GO_Slim_Categories"] = df["GO_Slim_Categories"].apply(_as_list)

# Plot variants: fill empty lists with sentinel so explode-then-filter is clean
df["Function_Slim_Plot"] = df["Function_Slim"].apply(lambda x: x if x else ["Unannotated"])
df["Location_Slim_Plot"] = df["Location_Slim"].apply(lambda x: x if x else ["Unannotated"])

# -- Raw GO IDs: kept separate, used only in the GO term filter section -------
df["GO_BP_list"] = df["GO_BP"].apply(_as_list)
df["GO_MF_list"] = df["GO_MF"].apply(_as_list)
df["GO_CC_list"] = df["GO_CC"].apply(_as_list)

print(f"{len(df):,} proteins  |  {df['pLLPS_Class'].value_counts().to_dict()}")
print(f"Membrane (TMD>0): {df['Is_Membrane'].sum():,}")
print(
    f"GO slim function terms: {df['Function_Slim'].explode().nunique():,} unique  |  "
    f"GO slim location terms: {df['Location_Slim'].explode().nunique():,} unique"
)
df.head(2)

In [ ]:
# Fig 1 -- pLLPS score distribution: count histogram + KDE density by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked count histogram
for cls in ["High", "Medium", "Low"]:
    s = df[df["pLLPS_Class"] == cls]["p(LLPS)"]
    axes[0].hist(s, bins=40, alpha=0.72, color=C[cls], label=f"{cls}  (n={len(s):,})")
axes[0].axvline(CUTOFF, color="#333333", linestyle="--", linewidth=1, alpha=0.6, label=f"cutoff {CUTOFF}")
axes[0].set(xlabel="p(LLPS)", ylabel="Count", title="pLLPS distribution by class")
axes[0].legend(title="Class")

# Right: normalised histogram + KDE overlay
for cls in ["High", "Medium", "Low"]:
    s = df[df["pLLPS_Class"] == cls]["p(LLPS)"]
    axes[1].hist(s, bins=40, density=True, alpha=0.28, color=C[cls])
    sns.kdeplot(s, ax=axes[1], color=C[cls], linewidth=2, label=cls)
axes[1].axvline(CUTOFF, color="#333333", linestyle="--", linewidth=1, alpha=0.6)
axes[1].set(xlabel="p(LLPS)", ylabel="Density", title="pLLPS density by class")
axes[1].legend(title="Class")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_pllps_distribution.png", dpi=300)
plt.show()

In [ ]:
# Fig 2 -- Membrane vs non-membrane: histogram overlay + violin/box
groups = {
    "Membrane":     df[df["Is_Membrane"]]["p(LLPS)"].dropna(),
    "Non-Membrane": df[~df["Is_Membrane"]]["p(LLPS)"].dropna(),
}
long   = pd.concat([s.rename("pLLPS").to_frame().assign(Group=k) for k, s in groups.items()])
_, p_mw = stats.mannwhitneyu(*groups.values(), alternative="two-sided")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping histograms + KDE
for label, s in groups.items():
    axes[0].hist(s, bins=40, density=True, alpha=0.45, color=C[label],
                 label=f"{label}  (n={len(s):,})")
    sns.kdeplot(s, ax=axes[0], color=C[label], linewidth=2)
axes[0].axvline(CUTOFF, color="#333333", linestyle="--", linewidth=1, alpha=0.6)
axes[0].set(xlabel="p(LLPS)", ylabel="Density", title="Histogram + KDE overlay")
axes[0].legend()

# Right: violin + box overlay
palette = {g: C[g] for g in groups}
sns.violinplot(data=long, x="Group", y="pLLPS", inner=None, cut=0,
               palette=palette, ax=axes[1], alpha=0.6)
sns.boxplot(data=long, x="Group", y="pLLPS", width=0.18, showfliers=False,
            palette=palette, ax=axes[1],
            boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.9},
            whiskerprops={"color": "#555555"}, capprops={"color": "#555555"},
            medianprops={"color": "#111111", "linewidth": 1.5})
axes[1].set(ylim=(0, 1.1), xlabel="", ylabel="p(LLPS)",
            title=f"Violin + box  (Mann-Whitney p = {p_mw:.2e})")

plt.suptitle("Membrane vs Non-Membrane pLLPS", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_membrane_vs_nonmembrane.png", dpi=300)
plt.show()

In [ ]:
# Fig 3 -- pLLPS histograms by top GO slim function categories
# Each panel: focal GO slim term (blue) vs all proteins (grey background).
func_long = (
    df[["p(LLPS)", "Function_Slim_Plot"]]
    .explode("Function_Slim_Plot")
    .rename(columns={"Function_Slim_Plot": "Function"})
    .query("Function not in ['Unannotated', 'Other', '']")
    .dropna(subset=["p(LLPS)"])
)
top_funcs = func_long["Function"].value_counts().nlargest(6).index.tolist()
s_all     = df["p(LLPS)"].dropna()

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
axes = axes.flatten()
for i, func in enumerate(top_funcs):
    ax = axes[i]
    s  = func_long.loc[func_long["Function"] == func, "p(LLPS)"]
    ax.hist(s_all, bins=40, density=True, alpha=0.15, color="#999999", label="All proteins")
    ax.hist(s,     bins=40, density=True, alpha=0.50, color=C["Membrane"], label=func)
    sns.kdeplot(s, ax=ax, color=C["Membrane"], linewidth=2)
    ax.axvline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=0.9, alpha=0.7)
    ax.set(title=f"{func}  (n={len(s):,})", xlabel="p(LLPS)", ylabel="Density")
    ax.legend(fontsize=8)

plt.suptitle("pLLPS by top GO slim function  (grey = all proteins)", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_pllps_by_go_slim_function.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 4 -- pLLPS by GO slim location (all proteins, top 10 by count)
loc_long = (
    df[["p(LLPS)", "Location_Slim_Plot"]]
    .explode("Location_Slim_Plot")
    .dropna(subset=["Location_Slim_Plot"])
    .query("Location_Slim_Plot not in ['Other', '', 'Unannotated']")
    .rename(columns={"Location_Slim_Plot": "Location"})
    .dropna(subset=["p(LLPS)"])
)
top_locs  = loc_long["Location"].value_counts().nlargest(TOP_N).index
loc_long  = loc_long[loc_long["Location"].isin(top_locs)]
order_locs = (
    loc_long.groupby("Location")["p(LLPS)"]
    .median().sort_values(ascending=False).index.tolist()
)

fig, ax = plt.subplots(figsize=(12, max(6, 0.42 * len(order_locs) + 1.5)))
sns.violinplot(data=loc_long, y="Location", x="p(LLPS)", order=order_locs,
               inner=None, cut=0, color=C["Cytosolic"], alpha=0.55, ax=ax)
sns.boxplot(data=loc_long, y="Location", x="p(LLPS)", order=order_locs,
            width=0.15, showfliers=False, ax=ax,
            boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.9},
            whiskerprops={"color": "#555555"}, capprops={"color": "#555555"},
            medianprops={"color": "#111111", "linewidth": 1.2})
ax.axvline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=0.9, alpha=0.6)
ax.set(xlim=(0, 1.1), xlabel="p(LLPS)", ylabel="",
       title=f"pLLPS by GO slim location -- all proteins  (top {TOP_N} by count)")
ax.tick_params(axis="y", labelsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig4_by_location.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 5 -- 2D KDE: pLLPS vs protein length and DPR count; violin/box vs TMD count
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# A: pLLPS vs log10(Length) -- 2D KDE with scatter underlay
s_len   = df.dropna(subset=["p(LLPS)", "Length"])
log_len = np.log10(s_len["Length"].clip(lower=1))
sns.kdeplot(x=log_len, y=s_len["p(LLPS)"], ax=axes[0],
            fill=True, cmap="Blues", thresh=0.04, levels=14)
axes[0].scatter(log_len, s_len["p(LLPS)"],
                alpha=0.015, s=4, color="#333333", rasterized=True)
axes[0].axhline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=1, alpha=0.7)
axes[0].set(xlabel="log\u2081\u2080(Length, aa)", ylabel="p(LLPS)",
            title=f"pLLPS vs protein length  (n={len(s_len):,})")

# B: pLLPS vs n(DPR>=25) -- 2D KDE, clip at 40 for readability
s_dpr = df.dropna(subset=["p(LLPS)", "n(DPR=> 25)"])
dpr   = s_dpr["n(DPR=> 25)"].clip(upper=40)
sns.kdeplot(x=dpr, y=s_dpr["p(LLPS)"], ax=axes[1],
            fill=True, cmap="Greens", thresh=0.04, levels=14)
axes[1].scatter(dpr, s_dpr["p(LLPS)"],
                alpha=0.015, s=4, color="#333333", rasterized=True)
axes[1].axhline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=1, alpha=0.7)
axes[1].set(xlabel="n(DPR >= 25)  [clipped at 40]", ylabel="p(LLPS)",
            title=f"pLLPS vs disordered region count  (n={len(s_dpr):,})")

# C: pLLPS by TMD count -- jittered scatter + violin + box overlay
tmd_grp   = df["TMD_count"].clip(upper=8).astype(str)
tmd_order = [str(i) for i in sorted(df["TMD_count"].clip(upper=8).unique())]
jitter    = rng.uniform(-0.25, 0.25, len(df))
axes[2].scatter(df["TMD_count"].clip(upper=8) + jitter, df["p(LLPS)"],
                alpha=0.03, s=4, color="#888888", rasterized=True)
sns.violinplot(data=df.assign(_tmd=tmd_grp), x="_tmd", y="p(LLPS)",
               order=tmd_order, inner=None, cut=0,
               color=C["Cytosolic"], alpha=0.5, ax=axes[2])
sns.boxplot(data=df.assign(_tmd=tmd_grp), x="_tmd", y="p(LLPS)",
            order=tmd_order, width=0.18, showfliers=False, ax=axes[2],
            boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.9},
            whiskerprops={"color": "#555555"}, capprops={"color": "#555555"},
            medianprops={"color": "#111111", "linewidth": 1.5})
axes[2].axhline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=1, alpha=0.7)
axes[2].set(xlabel="TMD count  (8 = 8+)", ylabel="p(LLPS)",
            title="pLLPS by TMD count  (violin + box)")

plt.suptitle("pLLPS vs structural features", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_2d_kde.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 6 -- Membrane proteins: pLLPS by GO slim function (violin + box, top 10 by count)
mem       = df[df["Is_Membrane"]].copy()
func_long = (
    mem[["p(LLPS)", "Function_Slim_Plot"]]
    .explode("Function_Slim_Plot")
    .dropna(subset=["Function_Slim_Plot"])
    .query("Function_Slim_Plot not in ['Other', '', 'Unannotated']")
    .rename(columns={"Function_Slim_Plot": "Function"})
    .dropna(subset=["p(LLPS)"])
)
func_counts = func_long["Function"].value_counts()
top_funcs   = func_counts.nlargest(TOP_N).index
func_long   = func_long[func_long["Function"].isin(top_funcs)].copy()
order_func  = func_counts.loc[top_funcs].sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(12, max(7, 0.35 * len(order_func) + 1.5)))
sns.violinplot(data=func_long, y="Function", x="p(LLPS)", order=order_func,
               inner=None, cut=0, color=C["Membrane"], alpha=0.55, ax=ax)
sns.boxplot(data=func_long, y="Function", x="p(LLPS)", order=order_func,
            width=0.15, showfliers=False, ax=ax,
            boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.9},
            whiskerprops={"color": "#555555"}, capprops={"color": "#555555"},
            medianprops={"color": "#111111", "linewidth": 1.2})
ax.axvline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=0.9, alpha=0.6)
ax.set(xlim=(0, 1.1), xlabel="p(LLPS)", ylabel="",
       title=f"Membrane proteins -- pLLPS by GO slim function  (top {TOP_N},  n={len(func_long):,} rows)")
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig6_membrane_by_function.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 7 -- Membrane proteins: pLLPS by GO slim location (violin + box, top 10, excl. cytosol)
loc_long = (
    mem[["p(LLPS)", "Location_Slim_Plot"]]
    .explode("Location_Slim_Plot")
    .dropna(subset=["Location_Slim_Plot"])
    .query("Location_Slim_Plot not in ['Other', '', 'Unannotated', 'cytosol']")
    .rename(columns={"Location_Slim_Plot": "Location"})
    .dropna(subset=["p(LLPS)"])
)
loc_counts = loc_long["Location"].value_counts()
top_locs   = loc_counts.nlargest(TOP_N).index
loc_long   = loc_long[loc_long["Location"].isin(top_locs)].copy()
order_locs = loc_counts.loc[top_locs].sort_values(ascending=False).index.tolist()

fig, ax = plt.subplots(figsize=(12, max(7, 0.42 * len(order_locs) + 1.5)))
sns.violinplot(data=loc_long, y="Location", x="p(LLPS)", order=order_locs,
               inner=None, cut=0, color=C["Cytosolic"], alpha=0.55, ax=ax)
sns.boxplot(data=loc_long, y="Location", x="p(LLPS)", order=order_locs,
            width=0.15, showfliers=False, ax=ax,
            boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.9},
            whiskerprops={"color": "#555555"}, capprops={"color": "#555555"},
            medianprops={"color": "#111111", "linewidth": 1.2})
ax.axvline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=0.9, alpha=0.6)
ax.set(xlim=(0, 1.1), xlabel="p(LLPS)", ylabel="",
       title=f"Membrane proteins -- pLLPS by GO slim location  (top {TOP_N}, excl. cytosol,  n={len(loc_long):,})")
ax.tick_params(axis="y", labelsize=10)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig7_membrane_by_location.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 8 -- % above p(LLPS) cutoff by GO slim function (membrane proteins, n > 10)
overall_pct = 100 * (mem["p(LLPS)"] > CUTOFF).sum() / len(mem)

func_stats = (
    mem[["Function_Slim_Plot", "p(LLPS)"]]
    .explode("Function_Slim_Plot")
    .dropna(subset=["Function_Slim_Plot"])
    .query("Function_Slim_Plot not in ['Other', '', 'Unannotated']")
    .groupby("Function_Slim_Plot")["p(LLPS)"]
    .agg(n="count", n_above=lambda s: (s > CUTOFF).sum())
    .reset_index()
)
func_stats["pct"] = 100 * func_stats["n_above"] / func_stats["n"]
func_stats_filt   = func_stats[func_stats["n"] > 10].sort_values("pct", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, 0.3 * len(func_stats_filt) + 1.5)))
ax.barh(func_stats_filt["Function_Slim_Plot"], func_stats_filt["pct"],
        color=C["Membrane"], alpha=0.85, edgecolor="white")
ax.axvline(overall_pct, color="#e74c3c", linestyle="--", linewidth=1.5,
           label=f"Membrane overall  {overall_pct:.1f}%")
for y, (_, row) in enumerate(func_stats_filt.iterrows()):
    ax.text(row["pct"] + 0.8, y, f"n={int(row['n'])}",
            va="center", fontsize=8, color="#333333")
ax.set(
    xlim=(0, max(100, func_stats_filt["pct"].max() * 1.12)),
    xlabel=f"% with p(LLPS) > {CUTOFF}", ylabel="",
    title=f"% above p(LLPS) {CUTOFF} by GO slim function  (membrane, n > 10)",
)
ax.tick_params(axis="y", labelsize=9)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig8_pct_above_cutoff_function.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Fig 9 -- % above p(LLPS) cutoff by GO slim location (membrane proteins, n > 10, excl. cytosol)
loc_stats = (
    mem[["Location_Slim_Plot", "p(LLPS)"]]
    .explode("Location_Slim_Plot")
    .dropna(subset=["Location_Slim_Plot"])
    .query("Location_Slim_Plot not in ['Other', '', 'Unannotated', 'cytosol']")
    .groupby("Location_Slim_Plot")["p(LLPS)"]
    .agg(n="count", n_above=lambda s: (s > CUTOFF).sum())
    .reset_index()
)
loc_stats["pct"] = 100 * loc_stats["n_above"] / loc_stats["n"]
loc_stats_filt   = loc_stats[loc_stats["n"] > 10].sort_values("pct", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, 0.42 * len(loc_stats_filt) + 1.5)))
ax.barh(loc_stats_filt["Location_Slim_Plot"], loc_stats_filt["pct"],
        color=C["Cytosolic"], alpha=0.85, edgecolor="white")
ax.axvline(overall_pct, color="#e74c3c", linestyle="--", linewidth=1.5,
           label=f"Membrane overall  {overall_pct:.1f}%")
for y, (_, row) in enumerate(loc_stats_filt.iterrows()):
    ax.text(row["pct"] + 0.8, y, f"n={int(row['n'])}",
            va="center", fontsize=8, color="#333333")
ax.set(
    xlim=(0, max(100, loc_stats_filt["pct"].max() * 1.12)),
    xlabel=f"% with p(LLPS) > {CUTOFF}", ylabel="",
    title=f"% above p(LLPS) {CUTOFF} by GO slim location  (membrane, n > 10, excl. cytosol)",
)
ax.tick_params(axis="y", labelsize=9)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig9_pct_above_cutoff_location.png", dpi=300, bbox_inches="tight")
plt.show()

---
## GO Term Filter

**Separate from GO slim categories above.**

Edit `FILTER_GO_TERMS` with raw GO IDs from any ontology (BP, MF, CC).  
Proteins match if they carry **any** of the listed terms in their full GO annotation — not limited to the slim subset.  
Set `FILTER_LABEL` to a short descriptive name for the titles.

In [ ]:
# -- GO TERM FILTER ----------------------------------------------------------
# Add GO IDs below to explore any specific subset.
# Independent of the GO slim groupings in Figs 3-9.
# -----------------------------------------------------------------------------

FILTER_GO_TERMS: list[str] = [
    # "GO:0005886",  # plasma membrane (CC)
    # "GO:0003723",  # RNA binding (MF)
    # "GO:0000785",  # chromatin (CC)
]
FILTER_LABEL = "GO term subset"   # short label for legend / titles

def _has_go_term(row: pd.Series, terms: list) -> bool:
    all_ids = row["GO_BP_list"] + row["GO_MF_list"] + row["GO_CC_list"]
    return any(t in all_ids for t in terms)

if not FILTER_GO_TERMS:
    print("No GO terms specified -- edit FILTER_GO_TERMS above and re-run.\n")
    print("Top GO IDs in dataset (by frequency):")
    for col, label in [("GO_BP_list", "BP"), ("GO_MF_list", "MF"), ("GO_CC_list", "CC")]:
        sample = df[col].explode().dropna().value_counts().head(5)
        print(f"  {label}: {sample.index.tolist()}")
else:
    mask   = df.apply(_has_go_term, axis=1, args=(FILTER_GO_TERMS,))
    subset = df[mask].copy()
    rest   = df[~mask].copy()
    print(f"Terms: {FILTER_GO_TERMS}")
    print(f"Matching: {mask.sum():,} / {len(df):,}  ({100 * mask.mean():.1f}%)")

    _, p_mw = stats.mannwhitneyu(
        subset["p(LLPS)"].dropna(), rest["p(LLPS)"].dropna(), alternative="two-sided"
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: histogram + KDE overlay
    for s, color, label in [
        (rest["p(LLPS)"].dropna(),   "#999999",     f"Other  (n={len(rest):,})"),
        (subset["p(LLPS)"].dropna(), C["Membrane"], f"{FILTER_LABEL}  (n={len(subset):,})"),
    ]:
        axes[0].hist(s, bins=40, density=True, alpha=0.4, color=color, label=label)
        sns.kdeplot(s, ax=axes[0], color=color, linewidth=2)
    axes[0].axvline(CUTOFF, color="#e74c3c", linestyle="--", linewidth=1, alpha=0.7)
    axes[0].set(xlabel="p(LLPS)", ylabel="Density",
                title=f"{FILTER_LABEL} vs rest  (Mann-Whitney p={p_mw:.2e})")
    axes[0].legend()

    # Right: violin + box overlay
    long_go = pd.concat([
        subset["p(LLPS)"].dropna().rename("pLLPS").to_frame().assign(Group=FILTER_LABEL),
        rest["p(LLPS)"].dropna().rename("pLLPS").to_frame().assign(Group="Other"),
    ])
    pal = {FILTER_LABEL: C["Membrane"], "Other": "#999999"}
    sns.violinplot(data=long_go, x="Group", y="pLLPS", inner=None, cut=0,
                   palette=pal, ax=axes[1], alpha=0.6)
    sns.boxplot(data=long_go, x="Group", y="pLLPS", width=0.18, showfliers=False,
                palette=pal, ax=axes[1],
                boxprops={"facecolor": "white", "edgecolor": "#333333", "linewidth": 0.9},
                whiskerprops={"color": "#555555"}, capprops={"color": "#555555"},
                medianprops={"color": "#111111", "linewidth": 1.5})
    axes[1].set(ylim=(0, 1.1), xlabel="", ylabel="p(LLPS)", title="Violin + box")

    plt.tight_layout()
    plt.show()

In [ ]:
print("Figures written to", FIG_DIR.resolve())
for f in sorted(FIG_DIR.glob("*.png")):
    print(" ", f.name)